# 01 - Data Exploration

Browse all tables in the MarketMind DuckDB database. Each section shows schema, sample rows, and key stats.

In [1]:
import duckdb
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

con = duckdb.connect('../data/marketmind.duckdb', read_only=True)
print("Tables:", [r[0] for r in con.sql("SELECT table_name FROM duckdb_tables()").fetchall()])
print("Views:", [r[0] for r in con.sql("SELECT view_name FROM duckdb_views() WHERE view_name IN ('market_summary','theme_summary','split_summary')").fetchall()])

Tables: ['markets_registry', 'snapshots_enriched', 'snapshots_raw', 'test', 'train', 'val']
Views: ['market_summary', 'split_summary', 'theme_summary']


## 1. markets_registry
95 curated binary markets — flat metadata from `configs/markets.yaml`.

In [2]:
con.sql("SELECT * FROM markets_registry").df()

,condition_id,question,slug,yes_token,volume_usd,resolved_yes,end_date,theme,theme_label,meeting_date,actual_outcome
0,0xa0811c97f529d627b7774a5b188e605736b745a1f892c39e16c5a022fdb84b8b,Fed rate cut by September 18?,fed-rate-cut-by-september-18,50376722767982976792819483321675476184683505970028931051190616149906273365875,20345318,True,2024-09-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp
1,0x6cc501fa617e3a46ececf6e3990fe1afeeaa6bc897d3c7049c005fd1bdc42c2f,Fed decreases interest rates by 50+ bps after September 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-september-2024-meeting,106191328358576540351439267765925450329859429577455659884974413809922495874408,10906333,True,2024-09-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp
2,0x70357debf15a956d82d8b2022aaf5f868763d1d0716d2c6e5ce35712e49a3b0f,Fed decreases interest rates by 25 bps after September 2024 meeting?,fed-decreases-interest-rates-by-25-bps-after-september-2024-meeting,88244443360063235221444316604590968694314258311386447899087521723508440858841,6660914,False,2024-09-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp
3,0x0a533c938200a10df86423dfbe621a2f36bfd4abc457615cf4992ed721c566bf,Fed increases interest rates by 25+ bps after September 2024 meeting?,fed-increases-interest-rates-by-25-bps-after-september-2024-meeting,95823178650727331613915203831778682038645976746731326695569990405131199144192,17625581,False,2024-09-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp
4,0xb20072387cd12bc73f3a6fd30fdd5aa1644c8f72f958803d90fc05c51db0ac49,No change in Fed interest rates after 2024 September meeting?,no-change-in-fed-interest-rates-after-2024-september-meeting,89262722133387845193166560202808972424089924545438804960915341631492994906283,23499211,False,2024-09-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp
...,...,...,...,...,...,...,...,...,...,...,...
90,0x55a469ea12fc89bf6b12c38fc764e35c05b4b520485adf4c8716d6369b38ad8a,Will the Government shutdown end November 12-15?,will-the-government-shutdown-end-november-12-15,7045107161367241233811523851106536676632348173555291268726302515224841822187,8422644,True,2025-11-15,government_shutdown,US government shutdown timing and duration markets,None,None
91,0xd5a91c9ee50ba80385283714f2a66b1e16d544a682af2af06a8f57fcf1d0233d,Will the government shutdown last 5 days or more?,will-the-government-shutdown-last-5-days-or-more,8008742846391096366381429238657491392199916771102623264605530168655505543184,6260856,False,2026-03-31,government_shutdown,US government shutdown timing and duration markets,None,None
92,0xd7b39f912814f41c7edc9a7693d54487611f4abee1ae4ab6cdf1a704be8a33d4,Will the government shutdown end November 13?,will-the-government-shutdown-end-november-13-182,17398404674846042629532943087095452701069816613172119438788363956594269928381,6006067,True,2025-11-21,government_shutdown,US government shutdown timing and duration markets,None,None
93,0xfa48a99317daef1654d5b03e30557c4222f276657275628d9475e141c64b545d,US recession in 2025?,us-recession-in-2025,104173557214744537570424345347209544585775842950109756851652855913015295701992,11734512,False,2026-02-28,recession,US recession probability markets,None,None


In [3]:
con.sql("""
    SELECT theme, count(*) as n_markets, 
           (sum(volume_usd)/1e6)::INT as volume_M,
           round(avg(CASE WHEN resolved_yes THEN 1.0 ELSE 0.0 END), 3) as yes_rate
    FROM markets_registry GROUP BY theme ORDER BY n_markets DESC
""").df()

,theme,n_markets,volume_M,yes_rate
0,fed_rate_decisions,56,2715,0.232
1,fed_leadership,21,578,0.048
2,government_shutdown,10,293,0.500
3,fed_rate_annual,6,54,0.333
4,tariffs_trade,1,11,1.000
5,recession,1,12,0.000


## 2. snapshots_raw
25,083 rows — one row per market per 12-hour price observation from the Polymarket API.

In [4]:
con.sql("SELECT * FROM snapshots_raw ORDER BY snapshot_ts LIMIT 20").df()

,condition_id,question,slug,theme,theme_label,meeting_date,actual_outcome,volume_usd,resolved_yes,end_date,snapshot_ts,price_yes,is_final_snapshot,snapshot_source
0,0xa0811c97f529d627b7774a5b188e605736b745a1f892c39e16c5a022fdb84b8b,Fed rate cut by September 18?,fed-rate-cut-by-september-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp,20345318,1,2024-09-18,2024-03-08 19:00:03-05:00,0.8250,False,api
1,0xa0811c97f529d627b7774a5b188e605736b745a1f892c39e16c5a022fdb84b8b,Fed rate cut by September 18?,fed-rate-cut-by-september-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp,20345318,1,2024-09-18,2024-03-09 07:00:03-05:00,0.8450,False,api
2,0xa0811c97f529d627b7774a5b188e605736b745a1f892c39e16c5a022fdb84b8b,Fed rate cut by September 18?,fed-rate-cut-by-september-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp,20345318,1,2024-09-18,2024-03-09 19:00:02-05:00,0.8450,False,api
3,0xa0811c97f529d627b7774a5b188e605736b745a1f892c39e16c5a022fdb84b8b,Fed rate cut by September 18?,fed-rate-cut-by-september-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp,20345318,1,2024-09-18,2024-03-10 08:00:03-04:00,0.8445,False,api
4,0xa0811c97f529d627b7774a5b188e605736b745a1f892c39e16c5a022fdb84b8b,Fed rate cut by September 18?,fed-rate-cut-by-september-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp,20345318,1,2024-09-18,2024-03-10 20:00:02-04:00,0.8445,False,api
5,0xa0811c97f529d627b7774a5b188e605736b745a1f892c39e16c5a022fdb84b8b,Fed rate cut by September 18?,fed-rate-cut-by-september-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp,20345318,1,2024-09-18,2024-03-11 08:00:02-04:00,0.8445,False,api
6,0xa0811c97f529d627b7774a5b188e605736b745a1f892c39e16c5a022fdb84b8b,Fed rate cut by September 18?,fed-rate-cut-by-september-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp,20345318,1,2024-09-18,2024-03-11 20:00:02-04:00,0.9050,False,api
7,0xa0811c97f529d627b7774a5b188e605736b745a1f892c39e16c5a022fdb84b8b,Fed rate cut by September 18?,fed-rate-cut-by-september-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp,20345318,1,2024-09-18,2024-03-12 08:00:02-04:00,0.9095,False,api
8,0xa0811c97f529d627b7774a5b188e605736b745a1f892c39e16c5a022fdb84b8b,Fed rate cut by September 18?,fed-rate-cut-by-september-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp,20345318,1,2024-09-18,2024-03-12 20:00:03-04:00,0.9150,False,api
9,0xa0811c97f529d627b7774a5b188e605736b745a1f892c39e16c5a022fdb84b8b,Fed rate cut by September 18?,fed-rate-cut-by-september-18,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-09-18,cut_50bp,20345318,1,2024-09-18,2024-03-13 08:00:03-04:00,0.9155,False,api


In [5]:
con.sql("""
    SELECT count(*) as rows, count(DISTINCT condition_id) as markets,
           min(snapshot_ts)::DATE as first_date, max(snapshot_ts)::DATE as last_date,
           round(avg(price_yes), 4) as avg_price, round(min(price_yes), 4) as min_price,
           round(max(price_yes), 4) as max_price
    FROM snapshots_raw
""").df()

,rows,markets,first_date,last_date,avg_price,min_price,max_price
0,25083,95,2024-03-08,2026-03-18,0.1692,0.0005,0.9995


## 3. snapshots_enriched
Same 25,083 rows + 10 engineered features (temporal, momentum, volatility).

In [6]:
con.sql("SELECT * FROM snapshots_enriched ORDER BY condition_id, snapshot_ts LIMIT 20").df()

,condition_id,question,slug,theme,theme_label,meeting_date,actual_outcome,volume_usd,resolved_yes,end_date,snapshot_ts,price_yes,is_final_snapshot,snapshot_source,days_to_end,days_to_meeting,pct_lifetime_elapsed,price_change,price_momentum_3,price_momentum_7,price_volatility_7,snapshot_num,total_snapshots,price_vs_open
0,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-06 20:00:01-04:00,0.315,False,api,132.999988,132.999988,0.000000,0.000,0.000000,0.000000,0.000000,1,268,0.000
1,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-07 08:00:01-04:00,0.320,False,api,132.499988,132.499988,0.003759,0.005,-0.002500,-0.002500,0.003536,2,268,0.005
2,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-07 20:00:01-04:00,0.330,False,api,131.999988,131.999988,0.007519,0.010,-0.008333,-0.008333,0.007638,3,268,0.015
3,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-08 08:00:01-04:00,0.335,False,api,131.499988,131.499988,0.011278,0.005,-0.006667,-0.010000,0.009129,4,268,0.020
4,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-08 20:00:02-04:00,0.335,False,api,130.999977,130.999977,0.015038,0.000,-0.001667,-0.008000,0.009083,5,268,0.020
5,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-09 08:00:01-04:00,0.085,False,api,130.499988,130.499988,0.018797,-0.250,0.166667,0.201667,0.099130,6,268,-0.230
6,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-09 20:00:02-04:00,0.090,False,api,129.999977,129.999977,0.022556,0.005,0.080000,0.168571,0.117108,7,268,-0.225
7,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-10 08:00:02-04:00,0.110,Fa

In [7]:
# Enriched feature distributions
con.sql("""
    SELECT 
        round(avg(days_to_end), 1) as avg_days_to_end,
        round(avg(pct_lifetime_elapsed), 3) as avg_pct_elapsed,
        round(avg(price_change), 5) as avg_price_chg,
        round(stddev(price_change), 5) as std_price_chg,
        round(avg(price_volatility_7), 5) as avg_vol7,
        round(avg(price_momentum_3), 5) as avg_mom3,
        round(avg(price_vs_open), 4) as avg_vs_open
    FROM snapshots_enriched
""").df()

,avg_days_to_end,avg_pct_elapsed,avg_price_chg,std_price_chg,avg_vol7,avg_mom3,avg_vs_open
0,177.0,0.408,0.00012,0.02248,0.01072,-0.00011,-0.0253


## 4. train / val / test
Modeling-ready splits with 6 extra columns: `market_price`, `theme_encoded`, `volume_rank`, `log_volume`, `days_to_end_bucket`, `num_outcomes`.

In [8]:
# Split overview
con.sql("""
    SELECT 'train' as split, count(*) as rows, count(DISTINCT condition_id) as markets,
           min(snapshot_ts)::DATE as first, max(snapshot_ts)::DATE as last,
           round(avg(resolved_yes), 3) as base_rate FROM train
    UNION ALL
    SELECT 'val', count(*), count(DISTINCT condition_id), min(snapshot_ts)::DATE, max(snapshot_ts)::DATE, round(avg(resolved_yes), 3) FROM val
    UNION ALL
    SELECT 'test', count(*), count(DISTINCT condition_id), min(snapshot_ts)::DATE, max(snapshot_ts)::DATE, round(avg(resolved_yes), 3) FROM test
""").df()

,split,rows,markets,first,last,base_rate
0,train,6469,32,2024-03-08,2024-12-31,0.288
1,val,5206,34,2024-12-31,2025-06-30,0.206
2,test,13408,54,2025-06-30,2026-03-18,0.126


In [9]:
# Browse train
con.sql("SELECT * FROM train LIMIT 20").df()

,condition_id,question,slug,theme,theme_label,meeting_date,actual_outcome,volume_usd,resolved_yes,end_date,snapshot_ts,price_yes,is_final_snapshot,snapshot_source,days_to_end,days_to_meeting,pct_lifetime_elapsed,price_change,price_momentum_3,price_momentum_7,price_volatility_7,snapshot_num,total_snapshots,price_vs_open,theme_encoded,volume_rank,log_volume,days_to_end_bucket,market_price,num_outcomes
0,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-06 20:00:01-04:00,0.315,False,api,132.999988,132.999988,0.000000,0.000,0.000000,0.000000,0.000000,1,268,0.000,2,0.183929,16.033878,3m-1y,0.315,2
1,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-07 08:00:01-04:00,0.320,False,api,132.499988,132.499988,0.003759,0.005,-0.002500,-0.002500,0.003536,2,268,0.005,2,0.183929,16.033878,3m-1y,0.320,2
2,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-07 20:00:01-04:00,0.330,False,api,131.999988,131.999988,0.007519,0.010,-0.008333,-0.008333,0.007638,3,268,0.015,2,0.183929,16.033878,3m-1y,0.330,2
3,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-08 08:00:01-04:00,0.335,False,api,131.499988,131.499988,0.011278,0.005,-0.006667,-0.010000,0.009129,4,268,0.020,2,0.183929,16.033878,3m-1y,0.335,2
4,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-08 20:00:02-04:00,0.335,False,api,130.999977,130.999977,0.015038,0.000,-0.001667,-0.008000,0.009083,5,268,0.020,2,0.183929,16.033878,3m-1y,0.335,2
5,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-09 08:00:01-04:00,0.085,False,api,130.499988,130.499988,0.018797,-0.250,0.166667,0.201667,0.099130,6,268,-0.230,2,0.183929,16.033878,3m-1y,0.085,2
6,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09f0864a4305,Fed decreases interest rates by 50 bps after December 2024 meeting?,fed-decreases-interest-rates-by-50-bps-after-december-2024-meeting,fed_rate_decisions,"FOMC meeting outcome markets — rate cuts, hikes, holds",2024-12-17 19:00:00-05:00,cut_25bp,9192310,0,2024-12-17 19:00:00-05:00,2024-08-09 20:00:02-04:00,0.090,False,api,129.999977,129.999977,0.022556,0.005,0.080000,0.168571,0.117108,7,268,-0.225,2,0.183929,16.033878,3m-1y,0.090,2
7,0x004e19d419d05dd366793689ce2af457d51bc921d9da4af867de09

In [ ]:
# Browse val
con.sql("SELECT * FROM val LIMIT 20").df()

In [ ]:
# Browse test
con.sql("SELECT * FROM test LIMIT 20").df()

## 5. Views — quick summaries

In [ ]:
# Per-market summary (all 95 markets)
con.sql("SELECT * FROM market_summary").df()

In [ ]:
# Theme summary
con.sql("SELECT * FROM theme_summary").df()

In [ ]:
# Theme distribution per split
con.sql("""
    SELECT theme,
        sum(CASE WHEN s='train' THEN n ELSE 0 END) as train,
        sum(CASE WHEN s='val' THEN n ELSE 0 END) as val,
        sum(CASE WHEN s='test' THEN n ELSE 0 END) as test
    FROM (
        SELECT theme, 'train' as s, count(*) as n FROM train GROUP BY theme
        UNION ALL SELECT theme, 'val', count(*) FROM val GROUP BY theme
        UNION ALL SELECT theme, 'test', count(*) FROM test GROUP BY theme
    ) GROUP BY theme ORDER BY train+val+test DESC
""").df()

## 6. Ad-hoc queries
Edit the SQL below to explore anything — DuckDB supports full SQL on all tables.

In [ ]:
# Edit this query to explore anything
con.sql("""
    SELECT * FROM snapshots_enriched
    WHERE theme = 'recession'
    ORDER BY snapshot_ts
    LIMIT 10
""").df()